In [15]:
from Bio import SeqIO
import math
import random
import numpy as np
import pandas as pd
from scipy.linalg import expm
import seaborn as sns 
import matplotlib.pyplot as plt
from itertools import combinations
from collections import Counter
from tqdm import tqdm

#set studentID as the random seed
random_seed = 2211284

In [16]:
print(os.listdir())

['.git', 'mutation_models_pipeline.ipynb', 'README.md', 'sequence.fasta.txt', 'tn93_longindel_parameter_priors.csv']


In [17]:
record = SeqIO.read("sequence.fasta","fasta")

#RegulonDB-based lac promoter window
start_1based = 365922
end_1based = 366435

#extract the genomic slice necessary
region = record.seq[start_1based - 1:end_1based]

#reverse complement because lacZp1 is on the reverse strand
lac_promoter = region.reverse_complement()

print("Length:",len(lac_promoter))
print(lac_promoter)

FileNotFoundError: [Errno 2] No such file or directory: 'sequence.fasta'

In [14]:
#define the hamming distance
def hamming_distance(seq1, seq2):
    assert(len(seq1) == len(seq2))
    #only valid for equal length sequences (susbtition-only mutation case)
    return sum(a != b for a, b in zip(seq1, seq2))

In [ ]:
#define the jukes cantor model
class Jukes_Cantor:
    
    def __init__(self):
        self.seed = random_seed

    def JC_single_base(self, base:str, mu:float, rng:random.Random) -> str:
        #Mutate a single DNA base under the Jukes-Cantor model

        #turn the base string into upper case
        base = base.upper()
        
        #probability of mutation
        p_same = 0.25 + 0.75*math.exp(-4.0*mu)

        if rng.random() < p_same:
            return base

        alternatives = [b for b in ["A", "C", "G", "T"] if b != base]

        random_num = rng.random()

        #mutate into another base with random probability
        if random_num < (1/3):
            return alternatives[0]
        elif random_num > (2/3):
            return alternatives[1]
        else:
            return alternatives[2]

    def JC_mutate_sequence(self, seq:str, mu:float, seed:int)->str:
        #Apply the JC mutation model independently across the DNA sequence
        rng = random.Random(seed)

        return "".join(self.JC_single_base(b, mu, rng) for b in seq)
    
    def generate_JC_library(self, seq:str, mu:float, n_samples:int) -> list[str]:
        #generate library of mutations from a single sequence
        rng = random.Random(self.seed)

        library = []

        for i in range(n_samples):

            #re-seed to keep randomness 
            child_seed = rng.randint(0,10**9)
            
            library.append(self.JC_mutate_sequence(seq,mu,child_seed))
        
        return library



In [ ]:
#defines the TN93 mutation model, that can be optionally extended for long insertion/deletion events
#this model can also be used for substitution-only events
class TN93_LongIndel_model:
    
    def __init__(self, kappa1:float, kappa2:float, base_frequencies:dict, mu_sub:float, mu_del:float, gamma:float, r: float):
        
        """
        Parameters
        ----------------------------------------------------------------------------------------------------------------------------------
        kappa1 : float
            Purine transition bias (A <-> G)
        kappa2 : float
            Pyrimidine transition bias (C <-> T)
        base_frequencies : dict
            Equilibrium nucleotide frequencies formatted as {"A":0.25,"C":0.25,"G":0.25,"T":0.25}
        mu_sub : float
            Substitution branch length / expected substitutions per site.
        mu_del : float
            Total deletion rate per site.
        gamma : float
            Controls insertion/deletion balance in the long indel model.
        r : float
            Geometric indel-length parameter, must satisfy 0 <= r < 1.
        """

        self.seed = random_seed
        self.kappa1 = kappa1
        self.kappa2 = kappa2

        self.base_frequencies = base_frequencies
        self.mu_sub = mu_sub
        self.mu_del = mu_del
        self.gamma = gamma
        self.r = r

        self.bases = ["A", "C", "G", "T"]
        self.base_to_index = {b:i for i,b in enumerate(self.bases)}

        self.Q = self.build_tn93_rate_matrix()

    #TN93 substitution section

    def transition_type(self, b1: str, b2: str) -> str:
        #purine transitiosn for A<->G, pyrimidine transitions C<->T, transversion otherwise
        if (b1, b2) in {("A","G"),("G","A")}:
            return "purine_transition"
        
        elif (b1, b2) in {("C","T"),("T","C")}:
            return "pyrimidine_transition"
        
        else:
            return "transversion"
    
    def build_tn93_rate_matrix(self) -> np.ndarray:

        #Construct the rate matrix Q 
        Q = np.zeros((4,4),dtype=float)

        for i, bi in enumerate(self.bases):
            for j, bj in enumerate(self.bases):
                if i == j:
                    #diagonal
                    continue 

                #find whether we are in the purine, pyrimidine transition or transversion case
                change_type = self.transition_type(bi,bj)

                #calculate mutation rates according to mutation type
                if change_type == "purine_transition":
                    Q[i, j] = self.kappa1 * self.base_frequencies[bj]

                elif change_type == "pyrimidine_transition":
                    Q[i, j] = self.kappa2 * self.base_frequencies[bj]

                else:
                    Q[i, j] = self.base_frequencies[bj]

        #set diagonals so rows sum to 0
        for i in range(4):
            Q[i,i] = -np.sum(Q[i,:])

        #rescale so the average substitution rate is 1
        pi_vec = np.array([self.base_frequencies[b] for b in self.bases])

        rate = -np.sum(pi_vec*np.diag(Q))
        Q /= rate

        return Q
    
    def get_transition_matrix(self, mu: float) -> np.ndarray:
        #compute P(u)
        return expm(self.Q * mu)

    def mutate_single_base(self,base: str, P: np.ndarray, rng:random.Random) -> str:
        #mutate a single DNA base
        base = base.upper()

        if base not in self.base_to_index:
            return base

        row_idx = self.base_to_index[base]
        probs = P[row_idx]

        probs = probs/probs.sum()

        u = rng.random()
        cumulative = 0.0

        for b, p in zip(self.bases, probs):

            cumulative += p

            if u <= cumulative:
                return b

        return self.bases[-1]
    
    def apply_substitutions(self, seq:str, rng:random.Random) -> str:
        #apply TN93 independently along each DNA sequence

        P = self.get_transition_matrix(self.mu_sub)
        return "".join(self.mutate_single_base(b, P, rng) for b in seq)
    
    #long indel section 

    def mu_k(self, k: int) -> float:

        #deletion rate for k-mer deletion
        return self.mu_del*(1-self.r)**2*(self.r**(k-1))

    def lambda_k(self, k: int) -> float:

        #Insertion rate for k-mer insertion
        return self.gamma * self.mu_del*(1-self.r)**2*((self.gamma*self.r)**(k-1))

    def sample_indel_length(self, mode: str, rng: random.Random, k_max: int = 20) -> int:

        #Sample k from the trundated indel-length distribution
        ks = np.arange(1, k_max+1)

        if mode == "del":
            weights = np.array([self.mu_k(int(k)) for k in ks],dtype=float)

        elif mode == "ins":
            weights = np.array([self.lambda_k(int(k)) for k in ks],dtype=float)

        else:
            raise ValueError("mode must be 'ins' or 'del'")

        weights = weights/weights.sum()

        return int(rng.choices(list(ks), weights=weights, k=1)[0])

    def sample_inserted_sequence(self, k:int,rng:random.Random) -> str:

        probs = [self.base_frequencies[b] for b in self.bases]
        return "".join(rng.choices(self.bases, weights=probs, k=k))

    def apply_deletions(self, seq: str, t: float, rng: random.Random, k_max: int = 20) -> str:
        #Approximate the number of deletion events as Poisson(L*num_del*t)

        seq_list = list(seq)
        L = len(seq_list)

        # print("Producing the number to delete")
        n_del = rng.poisson(L*self.mu_del*t) if hasattr(rng,"poisson") else np.random.poisson(L*self.mu_del*t)

        # print("Deleting")
        for i in range(n_del):
            if len(seq_list) == 0:
                break

            #find length of the deletion event
            k = self.sample_indel_length("del", rng, k_max=min(k_max,len(seq_list)))

            #just in case
            if k > len(seq_list):
                continue

            start = rng.randint(0, len(seq_list)-k)

            #delete base from selected position in list 
            del seq_list[start:start+k]

        return "".join(seq_list)

    def apply_insertions(self, seq: str, t: float, rng: random.Random, k_max: int = 20) -> str:
        #Approximate the number of insertion events as Poisson((L+1) * lambda_total * t)

        seq_list = list(seq)
        L = len(seq_list)

        lambda_total = sum(self.lambda_k(k) for k in range(1, k_max+1))

        # print("Producing the number to insert")
        n_ins = rng.poisson((L+1)*lambda_total*t) if hasattr(rng,"poisson") else np.random.poisson((L+1)*lambda_total*t)

        # print("Inserting")
        for io in range(n_ins):

            #find length of the insertion event
            k = self.sample_indel_length("ins", rng, k_max=k_max)

            inserted = list(self.sample_inserted_sequence(k,rng))

            pos = rng.randint(0, len(seq_list))

            #insert base into selected position in the list
            seq_list[pos:pos] = inserted

        # print("Returning from insertion")

        return "".join(seq_list)

    def mutate_sequence(self, seq:str, t:float = 1.0, k_max:int = 20, substitution_only:bool = False) -> str:

        #Apply substitutions, then deletions, the ninsertions
        rng = random.Random(self.seed)

        #use numpy RNG for poisson draws
        np.random.seed(self.seed)

        # print("Apply substitutions...")
        seq = self.apply_substitutions(seq,rng)

        # print("Applying insertions and deletions...")
        if not substitution_only:
            seq = self.apply_deletions(seq, t=t,rng=rng,k_max=k_max)
            seq = self.apply_insertions(seq, t=t,rng=rng,k_max=k_max)

        # print("Returning from mutating sequence")
        return seq

    def generate_library(self, seq: str, n_samples:int, t: float = 1.0, k_max: int = 20, substitution_only:bool = False) -> list[str]:

        #generate the library of substituion and insertion and deletion events based on the 'substitution_only' flag

        library = []
        master_rng = random.Random(self.seed)

        print("Generating a library of mutants")
        for i in tqdm(range(n_samples)):

            #maintain randomness
            child_seed = master_rng.randint(0, 10**9)
            self.seed = child_seed

            #add mutated sequences to the library
            mutated_sequence = self.mutate_sequence(seq, t=t, k_max=k_max, substitution_only=substitution_only)

            # print("Appending to the library")
            library.append(mutated_sequence)

        return library

In [ ]:
#sanity checking for one generation of mutations taking just one sample

JK_model = Jukes_Cantor()
mutants = JK_model.generate_JC_library(lac_promoter,mu=0.000,n_samples=1)

#The TN93 model collapses to the HKY85 model when kappa1=kappa2
#initialising random base frequencies
TN93_modell = TN93_LongIndel_model(kappa1=4.0,kappa2=1.0,base_frequencies={"A": 0.30,"C": 0.20,"G": 0.20,"T": 0.30}, mu_sub=0.00001, mu_del=0.000001, gamma=2, r=0.5)
mutantss = TN93_modell.generate_library(lac_promoter, n_samples=1)

print(lac_promoter[:30])

print("Jukes Cantor Model")

print(mutants[0][:30])

print("TN93 model")

print(mutantss[0][:30])

#This code takes approximately 1 minute to run

In [ ]:
def estimate_base_frequencies(seq:str) -> dict:
    #Estimate empirically the base frequencies from a given DNA sequence
    seq = seq.upper()

    #assuming the sequence does not contain any erroneous base frequencies
    valid_bases = [b for b in seq if b in {"A","C","G","T"}]

    counts = Counter(valid_bases)
    total = len(valid_bases)

    return {"A":counts.get("A",0)/total,"C": counts.get("C",0)/total,"G": counts.get("G",0)/total,"T": counts.get("T",0)/total}


# =========================================================
# Read parameter priors from CSV
# Expected CSV format:
# parameter,low,mode,high
# mu_sub,1.71e-10,1.99e-10,2.26e-10
# kappa1,1.0,1.3,2.0
# kappa2,1.0,1.6,2.5
# gamma,0.1,0.4,1.0
# r,0.05,0.10,0.20
# mu_del,1.09e-11,1.27e-11,1.45e-11
# =========================================================

def load_parameter_priors(filepath:str) -> pd.DataFrame:

    priors = pd.read_csv(filepath)

    required_cols = {"parameter", "low", "mode", "high"}

    missing = required_cols - set(priors.columns)

    if missing:
        raise ValueError(f"Parameter CSV is missing column {missing}")

    priors = priors.set_index("parameter")

    return priors

#sample on parameter set from the triangular prior found in the .csv file (per-site-per-generations rates/parameters)
def sample_tn93_longindel_parameters(lac_sequence: str,rng:random.Random,filepath: str)-> dict:

    base_frequencies = estimate_base_frequencies(lac_sequence)
    priors = load_parameter_priors(filepath)

    #extract the priors from the CSV file loaded dictionary
    mu_sub_row = priors.loc["mu_sub"]
    kappa1_row = priors.loc["kappa1"]
    kappa2_row = priors.loc["kappa2"]
    gamma_row = priors.loc["gamma"]
    r_row = priors.loc["r"]
    mu_del_row = priors.loc["mu_del"]

    #construct the traingular rate distributions from the priors
    mu_sub = rng.triangular(mu_sub_row["low"], mu_sub_row["high"],mu_sub_row["mode"])
    kappa1 = rng.triangular(kappa1_row["low"], kappa1_row["high"],kappa1_row["mode"])
    kappa2 = rng.triangular(kappa2_row["low"], kappa2_row["high"],kappa2_row["mode"])
    gamma = rng.triangular(gamma_row["low"], gamma_row["high"],gamma_row["mode"])
    r = rng.triangular(r_row["low"], r_row["high"],r_row["mode"])
    mu_del = rng.triangular(mu_del_row["low"], mu_del_row["high"],mu_del_row["mode"])

    return {"kappa1": kappa1,"kappa2": kappa2,"base_frequencies": base_frequencies,"mu_sub": mu_sub,"mu_del": mu_del, "gamma": gamma,"r": r}


#sample the parameter sets used to initialise the models based on the number of samples
def sample_parameter_sets(lac_sequence:str,n_samples:int,seed:int,filepath:str)-> list[dict]:

    rng = random.Random(seed)
    parameter_sets = []

    for i in range(n_samples):

        params = sample_tn93_longindel_parameters(lac_sequence=lac_sequence,rng=rng,filepath=filepath)

        parameter_sets.append(params)

    return parameter_sets


#function to summarise characteristics about mutant strains (hamming distance for substitution-only or length change for indel-enabled)
def summarise_mutant(reference_seq: str, mutant_seq: str, substitution_only: bool) -> dict:
    if substitution_only:

        n_substitutions = hamming_distance(reference_seq, mutant_seq)
        n_insertions = 0
        n_deletions = 0

    else:
        length_diff = len(mutant_seq)-len(reference_seq)
        n_insertions = max(length_diff,0)
        n_deletions = max(-length_diff,0)
        n_substitutions = np.nan

    return {"original_length": len(reference_seq),"mutant_length": len(mutant_seq),"n_substitutions": n_substitutions,"n_insertions": n_insertions,"n_deletions": n_deletions}


#Build libraries at specific generation horizons
def generate_mutation_libraries_across_generations(lac_sequence:str, parameter_csv_path:str, generation_horizons:list,n_mutants_per_generation:int,base_seed:int = 2211284,k_max: int = 4):
    #We generate two versions of mutation libraries for each generation horzion, one for substituion only, the other for indels and substitutions
    """
    Returns
    ------------------------------------------------------------------------------------
    substitution_only_df :pd.DataFrame
    substitution_plus_indel_df :pd.DataFrame
    """
    substitution_only_records = []
    substitution_plus_indel_records = []

    for gen in tqdm(generation_horizons):
        print(f"Generating libraries for {gen} generations")

        #sample one parameter set PER GENERATION HORIZON then scale mu_sub and mu_del by number of generations

        rng = random.Random(base_seed+int(gen))
        sampled_params = sample_tn93_longindel_parameters(lac_sequence=lac_sequence,rng=rng,filepath=parameter_csv_path)

        params_for_this_horizon = sampled_params.copy()
        params_for_this_horizon["mu_sub"] = sampled_params["mu_sub"]*gen
        params_for_this_horizon["mu_del"] = sampled_params["mu_del"]*gen

        #substituion-only ibrary
        sub_model = TN93_LongIndel_model(
                 kappa1=params_for_this_horizon["kappa1"],
                kappa2=params_for_this_horizon["kappa2"],
                base_frequencies=params_for_this_horizon["base_frequencies"],
                mu_sub=params_for_this_horizon["mu_sub"],
                mu_del=params_for_this_horizon["mu_del"],
                gamma=params_for_this_horizon["gamma"],
                r=params_for_this_horizon["r"])

        sub_library = sub_model.generate_library(seq=lac_sequence,n_samples=n_mutants_per_generation,t=1.0,k_max=k_max,substitution_only=True)

        for i, mutant_seq in enumerate(sub_library):
            summary = summarise_mutant(reference_seq=lac_sequence,mutant_seq=mutant_seq,substitution_only=True)

            substitution_only_records.append({
                    "library_type": "substitution_only",
                    "generations": gen,
                    "mutant_id": f"sub_only_g{gen}_{i}",
                    "sequence":mutant_seq,
                    "kappa1": params_for_this_horizon["kappa1"],
                    "kappa2": params_for_this_horizon["kappa2"],
                    "mu_sub_total": params_for_this_horizon["mu_sub"],
                    "mu_del_total": params_for_this_horizon["mu_del"],
                    "gamma": params_for_this_horizon["gamma"],
                    "r":params_for_this_horizon["r"],
                    **summary})

        #substitution and indel library
        indel_model = TN93_LongIndel_model(
                kappa1=params_for_this_horizon["kappa1"],
                kappa2=params_for_this_horizon["kappa2"],
                base_frequencies=params_for_this_horizon["base_frequencies"],
                mu_sub=params_for_this_horizon["mu_sub"],
                mu_del=params_for_this_horizon["mu_del"],
                gamma=params_for_this_horizon["gamma"],
                r=params_for_this_horizon["r"])

        indel_library = indel_model.generate_library(seq=lac_sequence,n_samples=n_mutants_per_generation,t=1.0,k_max=k_max,substitution_only=False)

        for i, mutant_seq in enumerate(indel_library):
            summary = summarise_mutant(reference_seq=lac_sequence,mutant_seq=mutant_seq,substitution_only=False)

            substitution_plus_indel_records.append({
                    "library_type": "substitution_plus_indel",
                    "generations": gen,
                    "mutant_id": f"sub_indel_g{gen}_{i}",
                    "sequence": mutant_seq,
                    "kappa1": params_for_this_horizon["kappa1"],
                    "kappa2": params_for_this_horizon["kappa2"],
                    "mu_sub_total": params_for_this_horizon["mu_sub"],
                    "mu_del_total": params_for_this_horizon["mu_del"],
                    "gamma": params_for_this_horizon["gamma"],
                    "r": params_for_this_horizon["r"],
                    **summary})

    substitution_only_df = pd.DataFrame(substitution_only_records)
    substitution_plus_indel_df = pd.DataFrame(substitution_plus_indel_records)

    return substitution_only_df,substitution_plus_indel_df

In [ ]:
#one mutation model for one lineage repeated many times to make a library (population size is effectively 1 so unlikely to product many mutations)

#ten generations of bacteria (typically for a 24 hour batch culture)
generation_horizons = [10_000_000, 100_000_000, 1_000_000_000]
parameter_csv_path = "tn93_longindel_parameter_priors.csv"

#define the number of independent lineages (the population of the agents being modelled)
#population of 10^6
number_lineages = 10_000

sub_only_df, sub_indel_df = generate_mutation_libraries_across_generations(lac_sequence=lac_promoter,parameter_csv_path=parameter_csv_path,generation_horizons=generation_horizons,n_mutants_per_generation=number_lineages,base_seed=random_seed,k_max=4)

NameError: name 'generate_mutation_libraries_across_generations' is not defined

In [ ]:
sub_only_df.to_csv("lac_promoter_substitution_only_libraries.csv", index=False)
sub_indel_df.to_csv("lac_promoter_substitution_plus_indel_libraries.csv", index=False)

In [ ]:
print(sub_only_df.head())
print(sub_indel_df.head())

print(sub_only_df.groupby("generations").size())
print(sub_indel_df.groupby("generations").size())

In [ ]:
#Plotting the number of substitutions over generations 
plt.figure(figsize=(10,10))
sns.boxplot(data=sub_only_df,x="generations",y="n_substitutions")
plt.title("Substitions accumulated vs generations")
plt.ylabel("Number of substitutions")
plt.xlabel("Generation horizon")
plt.show()

#Plotting the sequence length change over generations 
plt.figure(figsize=(10,10))
sns.boxplot(data=sub_indel_df,x="generations",y="mutant_length")
plt.title("Sequence Length Distribution under indel model")
plt.ylabel('Mutant Sequence Length')
plt.xlabel('Generation Horizon')
plt.show()

#Visualise the shape of the mutatnt distribution in a histogram 
plt.figure(figsize=(10,10))
sns.histplot(sub_only_df["n_substitutions"],bins=30,kde=True)
plt.title("Distribution of substitutions in substitution-only library")
plt.xlabel("Number of substitutions")
plt.show()

#Plot the pariwise Hamming distances between mutants
def average_pairwise_distance(seqs, sample_size = 100):

    seqs = seqs[:sample_size]
    dists = []

    for s1, s2 in combinations(seqs, 2):

        if len(s1) == len(s2):

            d = sum(a!=b for a,b in zip(s1,s2))
            dists.append(d)

    return np.mean(dists)

diversity = []

for g in sub_only_df["generations"].unique():

    seqs = sub_only_df[sub_only_df["generations"]==g]["sequence"].tolist()
    
    d = average_pairwise_distance(seqs)
    diversity.append((g,d))

div_df = pd.DataFrame(diversity, columns=["generations","avg_distance"])

plt.figure(figsize=(10,10))
plt.plot(div_df["generations"], div_df["avg_distance"],marker="o")
plt.xscale("log")
plt.xlabel("Generations")
plt.ylabel("Average pairwise distance")
plt.title("Sequence diversity vs evolutionary time")
plt.show()